In [ ]:
# === CELL 1: Environment Auto-Detection & Logging
# ==============================================================================

import os, sys, gc, re, time, warnings, json
import numpy as np
import pandas as pd
from collections import defaultdict
warnings.filterwarnings('ignore')

IS_KAGGLE = os.path.exists("/kaggle")
IS_AWS = os.path.exists("/home/ubuntu") or not IS_KAGGLE

if IS_KAGGLE:
    ENV_NAME = "Kaggle (2x T4 GPU, 30GB RAM)"
    KAGGLE_INPUT = "/kaggle/input/datasets/tatwiksaimolleti/amazon-ml-challenge-2026"
    if os.path.exists(os.path.join(KAGGLE_INPUT, "test")):
        TEST_DIR = os.path.join(KAGGLE_INPUT, "test")
    elif os.path.exists(os.path.join(KAGGLE_INPUT, "dataset", "test")):
        TEST_DIR = os.path.join(KAGGLE_INPUT, "dataset", "test")
    else:
        TEST_DIR = KAGGLE_INPUT
    OUTPUT_DIR = "/kaggle/working"
else:
    ENV_NAME = "AWS EC2 / Standalone Server"
    BASE_DIR = os.path.expanduser("~/data") if os.path.exists(os.path.expanduser("~/data")) else os.getcwd()
    TEST_DIR = os.path.join(BASE_DIR, "test") if os.path.exists(os.path.join(BASE_DIR, "test")) else BASE_DIR
    OUTPUT_DIR = os.path.join(BASE_DIR, "output_v4")

os.makedirs(OUTPUT_DIR, exist_ok=True)
NUM_CPUS = os.cpu_count() or 4

print("=" * 80)
print(f"🚀 TEST INFERENCE PIPELINE INITIALIZING ON: {ENV_NAME}")
print(f"   CPUs Available: {NUM_CPUS} cores")
print(f"   Test Directory: {TEST_DIR}")
print(f"   Output Folder:  {OUTPUT_DIR}")
print("=" * 80)

# Automatically clean obsolete heavy training embeddings to free 15.2 GB disk space
for _old_f in ["train_s2s3_emb.npy", "train_s2s3_emb.npy.progress", "train_sub_s1_emb.npy"]:
    _old_p = os.path.join(OUTPUT_DIR, _old_f)
    if os.path.exists(_old_p):
        try:
            os.remove(_old_p)
            print(f"🗑️ Reclaimed disk: removed {_old_f} (freed space for test embeddings)")
        except Exception as _e:
            pass



def fmt_time(seconds):
    if seconds is None or seconds < 0:
        return "0s"
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    if h > 0:
        return f"{h}h {m:02d}m {s:02d}s"
    elif m > 0:
        return f"{m}m {s:02d}s"
    return f"{s}s"


# ==============================================================================

In [ ]:
# === CELL 2: Package Verification & High-Performance Dependencies
# ==============================================================================

import subprocess

required_pkgs = ["rapidfuzz", "sparse_dot_topn", "unidecode", "lightgbm", "sentence-transformers", "torch"]
for pkg in required_pkgs:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        print(f"  Installing missing package: {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import faiss
except ImportError:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-gpu"])
        import faiss
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])
        import faiss

import torch
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
from sklearn.feature_extraction.text import TfidfVectorizer
from unidecode import unidecode
import lightgbm as lgb
from sentence_transformers import SentenceTransformer

try:
    from sparse_dot_topn import sp_matmul_topn
    USE_NEW_API = True
except ImportError:
    from sparse_dot_topn import awesome_cossim_topn
    USE_NEW_API = False

CUDA_AVAILABLE = torch.cuda.is_available()
GPU_COUNT = torch.cuda.device_count() if CUDA_AVAILABLE else 0
print(f"  PyTorch Version: {torch.__version__} | CUDA: {CUDA_AVAILABLE} (Found {GPU_COUNT} GPUs)")
for i in range(GPU_COUNT):
    print(f"    GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)")

FAISS_GPUS = faiss.get_num_gpus() if hasattr(faiss, 'get_num_gpus') else 0
print(f"  FAISS GPU acceleration: {'✅ Available (' + str(FAISS_GPUS) + ' GPUs)' if FAISS_GPUS > 0 else '⚠️ CPU fallback'}")
print("✅ All dependencies verified!")


# ==============================================================================

In [ ]:
# === CELL 3: Preprocessing & Text Normalization Functions
# ==============================================================================

def find_file(filename):
    candidates = [
        os.path.join(TEST_DIR, filename),
        os.path.join(OUTPUT_DIR, filename),
        filename
    ]
    if IS_KAGGLE:
        candidates.append(os.path.join("/kaggle/input", filename))
        # Search all subdirectories of /kaggle/input
        for root, _, files in os.walk("/kaggle/input"):
            if filename in files:
                return os.path.join(root, filename)
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Cannot find {filename} across paths.")


def normalize_name(name):
    if pd.isna(name) or str(name).strip() == "":
        return ""
    s = unidecode(str(name)).lower().strip()
    s = re.sub(r'^[-\s\.\*#]+', '', s)
    legal_patterns = [
        (r'\bincorporated\b', 'inc'), (r'\bcorporation\b', 'corp'), (r'\blimited\b', 'ltd'),
        (r'\bprivate\b', 'pvt'), (r'\bcompany\b', 'co'), (r'\bllc\b', 'llc'),
        (r'\bl\.l\.c\.?\b', 'llc'), (r'\bllp\b', 'llp'), (r'\bl\.l\.p\.?\b', 'llp'),
        (r'\binc\.?\b', 'inc'), (r'\bltd\.?\b', 'ltd'), (r'\bpvt\.?\b', 'pvt'),
        (r'\bcorp\.?\b', 'corp'), (r'\bco\.?\b', 'co'),
        (r'\bs\.?a\.?s\.?u\.?\b', 'sasu'), (r'\bs\.?a\.?s\.?\b', 'sas'),
        (r'\bs\.?a\.?r\.?l\.?\b', 'sarl'), (r'\be\.?u\.?r\.?l\.?\b', 'eurl'),
        (r'\bs\.?a\.?\b', 'sa'), (r'\bs\.?c\.?i\.?\b', 'sci'),
    ]
    for pat, rep in legal_patterns:
        s = re.sub(pat, rep, s)
    s = s.replace('&', ' and ')
    s = re.sub(r'[^\w\s]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()


def normalize_address(addr):
    if pd.isna(addr) or str(addr).strip().lower() in ("", "nan", "null", "none", "n/a"):
        return ""
    s = unidecode(str(addr)).lower().strip()
    addr_patterns = [
        (r'\bst\b', 'street'), (r'\brd\b', 'road'), (r'\bave\b', 'avenue'),
        (r'\bblvd\b', 'boulevard'), (r'\bdr\b', 'drive'), (r'\bct\b', 'court'),
        (r'\bln\b', 'lane'), (r'\bpkwy\b', 'parkway'), (r'\bhwy\b', 'highway'),
        (r'\bapt\b', 'apartment'), (r'\bste\b', 'suite'),
    ]
    for pat, rep in addr_patterns:
        s = re.sub(pat, rep, s)
    s = re.sub(r'\bnull\b', '', s)
    s = re.sub(r'[^\w\s]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()


def preprocess_df(df, name=""):
    t0 = time.time()
    print(f"  Preprocessing {name} ({len(df):,} records)...", end=" ", flush=True)
    df = df.copy()
    if 'country' in df.columns:
        df['country'] = df['country'].fillna('Unknown').astype(str).str.strip()
    else:
        df['country'] = 'Unknown'
    df['name_norm'] = df['business_name'].apply(normalize_name)
    df['addr_norm'] = df['business_address'].apply(normalize_address)
    df['name_addr'] = (df['name_norm'] + ' ' + df['addr_norm']).str.strip()
    raw_name = df['business_name'].fillna('').astype(str).str.strip()
    raw_addr = df['business_address'].fillna('').astype(str).str.strip()
    df['embed_text'] = (raw_name + ' ' + raw_addr).str.strip()
    print(f"done in {time.time()-t0:.1f}s")
    return df


# ==============================================================================

In [ ]:
# === CELL 4: Multilingual Embedding Engine & Candidate Blocking Functions
# ==============================================================================

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
EMBED_DEVICE = 'cuda:0' if CUDA_AVAILABLE else 'cpu'

print(f"\nInitializing Multilingual Embedding Model ({MODEL_NAME}) on {EMBED_DEVICE}...")
embed_model = SentenceTransformer(MODEL_NAME, device=EMBED_DEVICE)
print(f"✅ Model ready. Dimension: {embed_model.get_sentence_embedding_dimension()}")


def encode_texts_to_disk(texts, output_path, desc="texts", batch_size=256, max_len=200):
    n_total = len(texts)
    _dim = embed_model.get_sentence_embedding_dimension()
    t0 = time.time()
    os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)
    progress_file = output_path + ".progress"
    CHUNK_SIZE = 500000
    n_chunks = (n_total + CHUNK_SIZE - 1) // CHUNK_SIZE

    if os.path.exists(output_path) and not os.path.exists(progress_file):
        try:
            cached = np.load(output_path, mmap_mode='r')
            if cached.shape == (n_total, _dim):
                print(f"  ✅ [Cache hit] {desc} already completed: {output_path} (shape {cached.shape})", flush=True)
                return cached
        except Exception:
            pass

    start_chunk = 0
    if os.path.exists(output_path) and os.path.exists(progress_file):
        try:
            with open(progress_file, 'r') as f:
                start_chunk = int(f.read().strip())
            mmap_arr = np.lib.format.open_memmap(output_path, mode='r+')
            if mmap_arr.shape == (n_total, _dim) and start_chunk < n_chunks:
                print(f"  🔄 Resuming {desc} from Chunk {start_chunk + 1}/{n_chunks}...", flush=True)
            else:
                start_chunk = 0
                mmap_arr = np.lib.format.open_memmap(output_path, mode='w+', dtype=np.float32, shape=(n_total, _dim))
        except Exception:
            start_chunk = 0
            mmap_arr = np.lib.format.open_memmap(output_path, mode='w+', dtype=np.float32, shape=(n_total, _dim))
    else:
        mmap_arr = np.lib.format.open_memmap(output_path, mode='w+', dtype=np.float32, shape=(n_total, _dim))
        with open(progress_file, 'w') as f:
            f.write('0')

    if GPU_COUNT >= 2:
        target_devices = [f'cuda:{i}' for i in range(GPU_COUNT)]
        print(f"  Encoding {desc}: {n_total:,} items | {GPU_COUNT} GPUs {target_devices} | {n_chunks} chunks | to disk", flush=True)
        try:
            pool = embed_model.start_multi_process_pool(target_devices=target_devices)
            for ci in range(start_chunk, n_chunks):
                c_start = ci * CHUNK_SIZE
                c_end = min(c_start + CHUNK_SIZE, n_total)
                t_chunk = time.time()
                chunk_raw = texts[c_start:c_end]
                if hasattr(chunk_raw, 'tolist'):
                    chunk_raw = chunk_raw.tolist()
                chunk_texts = [t[:max_len] if len(t) > max_len else t for t in chunk_raw]
                del chunk_raw
                chunk_emb = embed_model.encode_multi_process(chunk_texts, pool, batch_size=batch_size)
                norms = np.linalg.norm(chunk_emb, axis=1, keepdims=True)
                mmap_arr[c_start:c_end] = (chunk_emb / np.maximum(norms, 1e-8)).astype(np.float32)
                mmap_arr.flush()
                with open(progress_file, 'w') as f:
                    f.write(str(ci + 1))
                del chunk_texts, chunk_emb, norms
                gc.collect()
                elapsed = time.time() - t0
                rate = c_end / max(elapsed, 0.001)
                eta = (n_total - c_end) / max(rate, 1)
                pct = c_end / n_total * 100
                print(f"    [{desc}] Chunk {ci+1:2d}/{n_chunks}: {c_end:9,}/{n_total:,} ({pct:5.1f}%) | {rate:5.0f}/s | Chunk: {time.time()-t_chunk:.0f}s | ETA: {fmt_time(eta)}", flush=True)
            embed_model.stop_multi_process_pool(pool)
            del mmap_arr
            gc.collect()
            if os.path.exists(progress_file):
                os.remove(progress_file)
            print(f"  ✅ {desc} done (dual-GPU): shape ({n_total}, {_dim}) in {fmt_time(time.time()-t0)}")
            return np.load(output_path, mmap_mode='r')
        except Exception as e:
            print(f"  Dual-GPU fallback: {e}")

    print(f"  Encoding {desc}: {n_total:,} items (single-device)...", flush=True)
    for ci in range(start_chunk, n_chunks):
        c_start = ci * CHUNK_SIZE
        c_end = min(c_start + CHUNK_SIZE, n_total)
        t_chunk = time.time()
        chunk_raw = texts[c_start:c_end]
        if hasattr(chunk_raw, 'tolist'):
            chunk_raw = chunk_raw.tolist()
        chunk_texts = [t[:max_len] if len(t) > max_len else t for t in chunk_raw]
        del chunk_raw
        chunk_emb = embed_model.encode(chunk_texts, batch_size=batch_size, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True)
        mmap_arr[c_start:c_end] = chunk_emb.astype(np.float32)
        mmap_arr.flush()
        with open(progress_file, 'w') as f:
            f.write(str(ci + 1))
        del chunk_texts, chunk_emb
        gc.collect()
        elapsed = time.time() - t0
        rate = c_end / max(elapsed, 0.001)
        eta = (n_total - c_end) / max(rate, 1)
        pct = c_end / n_total * 100
        print(f"    [{desc}] Chunk {ci+1:2d}/{n_chunks}: {c_end:9,}/{n_total:,} ({pct:5.1f}%) | {rate:5.0f}/s | ETA: {fmt_time(eta)}", flush=True)
    del mmap_arr
    gc.collect()
    if os.path.exists(progress_file):
        os.remove(progress_file)
    print(f"  ✅ {desc} done in {fmt_time(time.time()-t0)}")
    return np.load(output_path, mmap_mode='r')


def sparse_topn(A, B_T, top_n, threshold, n_threads=NUM_CPUS):
    if USE_NEW_API:
        return sp_matmul_topn(A, B_T, top_n=top_n, threshold=threshold, n_threads=n_threads)
    else:
        return awesome_cossim_topn(A, B_T, ntop=top_n, lower_bound=threshold, use_threads=True, n_jobs=n_threads)


def tfidf_query_blocking(s1_df, s2s3_df, field='name_norm', top_k=8, threshold=0.20):
    n_s1 = len(s1_df)
    n_s2s3 = len(s2s3_df)
    if n_s1 == 0 or n_s2s3 == 0:
        return []
    t0 = time.time()
    s1_texts = s1_df[field].fillna('').tolist()
    s2s3_texts = s2s3_df[field].fillna('').tolist()
    s1_ids = s1_df['entity_id'].values
    s2s3_ids = s2s3_df['entity_id'].values

    vectorizer = TfidfVectorizer(
        analyzer='char_wb', ngram_range=(3, 4), min_df=2,
        max_features=120000, dtype=np.float32, sublinear_tf=True
    )
    print(f"      [{field}] Fitting TF-IDF on {n_s1:,} S1 queries...", end=" ", flush=True)
    s1_tfidf = vectorizer.fit_transform(s1_texts)
    print(f"done. Vocab: {len(vectorizer.vocabulary_):,}")

    chunk_size = 500000
    n_chunks = (n_s2s3 + chunk_size - 1) // chunk_size
    print(f"      [{field}] Querying {n_s2s3:,} S2+S3 records in {n_chunks} chunks...")
    acc_s1, acc_s2s3, acc_score = [], [], []

    for ci in range(n_chunks):
        start = ci * chunk_size
        end = min(start + chunk_size, n_s2s3)
        chunk_tfidf = vectorizer.transform(s2s3_texts[start:end])
        cos_matrix = sparse_topn(s1_tfidf, chunk_tfidf.T, top_n=top_k, threshold=threshold, n_threads=NUM_CPUS)
        coo = cos_matrix.tocoo()
        if len(coo.data) > 0:
            acc_s1.append(coo.row.astype(np.int32))
            acc_s2s3.append((coo.col + start).astype(np.int32))
            acc_score.append(coo.data.astype(np.float32))
        del chunk_tfidf, cos_matrix, coo
        gc.collect()

    del s1_tfidf
    gc.collect()
    if not acc_s1:
        return []
    cat_s1 = np.concatenate(acc_s1)
    cat_s2s3 = np.concatenate(acc_s2s3)
    cat_score = np.concatenate(acc_score)

    if n_chunks > 1:
        df_merge = pd.DataFrame({'s1': cat_s1, 's2s3': cat_s2s3, 'sc': cat_score})
        df_merge = df_merge.sort_values('sc', ascending=False).groupby('s1').head(top_k).reset_index(drop=True)
        res_s1, res_s2s3, res_sc = df_merge['s1'].values, df_merge['s2s3'].values, df_merge['sc'].values
        del df_merge
    else:
        res_s1, res_s2s3, res_sc = cat_s1, cat_s2s3, cat_score

    candidates = list(zip(s1_ids[res_s1], s2s3_ids[res_s2s3], res_sc.astype(float)))
    print(f"      [{field}] → {len(candidates):,} pairs in {fmt_time(time.time()-t0)}")
    return candidates


def faiss_gpu_blocking(s1_df, s1_emb, s2s3_df, s2s3_emb, top_k=8, threshold=0.35):
    n_s1 = len(s1_df)
    n_s2s3 = len(s2s3_df)
    if n_s1 == 0 or n_s2s3 == 0:
        return []
    t0 = time.time()
    s1_ids = s1_df['entity_id'].values
    s2s3_ids = s2s3_df['entity_id'].values
    dim = s1_emb.shape[1]
    MAX_INDEX_SIZE = 1500000
    use_gpu = FAISS_GPUS > 0 and CUDA_AVAILABLE
    gpu_label = f"⚡ Dual-GPU ({GPU_COUNT}x T4)" if (use_gpu and GPU_COUNT >= 2) else ("⚡ GPU" if use_gpu else "💻 CPU fallback")
    print(f"      [FAISS Dense] Nearest-Neighbor Search using {gpu_label}...")

    def _build_index(emb_chunk):
        idx_cpu = faiss.IndexFlatIP(dim)
        idx_cpu.add(emb_chunk.astype(np.float32))
        if use_gpu:
            try:
                if GPU_COUNT >= 2:
                    co = faiss.GpuMultipleClonerOptions()
                    co.shard = False
                    co.useFloat16 = True
                    gpu_idx = faiss.index_cpu_to_all_gpus(idx_cpu, co=co)
                    del idx_cpu
                    gc.collect()
                    return gpu_idx
                else:
                    res = faiss.StandardGpuResources()
                    gpu_idx = faiss.index_cpu_to_gpu(res, 0, idx_cpu)
                    del idx_cpu
                    gc.collect()
                    return gpu_idx
            except Exception as e:
                print(f"  (FAISS GPU fallback: {e})", flush=True)
        faiss.omp_set_num_threads(NUM_CPUS)
        return idx_cpu

    q_batch_size = 20000
    n_q_batches = (n_s1 + q_batch_size - 1) // q_batch_size
    candidates = []
    s2s3_chunk_size = MAX_INDEX_SIZE if n_s2s3 > MAX_INDEX_SIZE else n_s2s3
    n_s2s3_chunks = (n_s2s3 + s2s3_chunk_size - 1) // s2s3_chunk_size

    for s2s3_ci in range(n_s2s3_chunks):
        s2s3_start = s2s3_ci * s2s3_chunk_size
        s2s3_end = min(s2s3_start + s2s3_chunk_size, n_s2s3)
        s2s3_ids_chunk = s2s3_ids[s2s3_start:s2s3_end]
        s2s3_emb_chunk = s2s3_emb[s2s3_start:s2s3_end]
        index = _build_index(s2s3_emb_chunk)
        del s2s3_emb_chunk

        for qb in range(n_q_batches):
            q_start = qb * q_batch_size
            q_end = min(q_start + q_batch_size, n_s1)
            q_vecs = s1_emb[q_start:q_end]
            scores, nn_indices = index.search(q_vecs, top_k)
            for i in range(len(q_vecs)):
                s1_global_idx = q_start + i
                for k in range(scores.shape[1]):
                    sim = float(scores[i, k])
                    t_idx = int(nn_indices[i, k])
                    if t_idx >= 0 and sim >= threshold:
                        candidates.append((s1_ids[s1_global_idx], s2s3_ids_chunk[t_idx], sim))
        del index
        gc.collect()
        if CUDA_AVAILABLE:
            torch.cuda.empty_cache()

    print(f"      [FAISS Dense] → {len(candidates):,} pairs in {fmt_time(time.time()-t0)}")
    return candidates


def generate_candidate_universe(s1_df, s1_emb, s2s3_df, s2s3_emb, max_candidates_per_s1=10):
    t_start = time.time()
    countries = s1_df['country'].dropna().unique()
    print(f"  Partitioning blocking across countries: {list(countries)}")
    country_dfs = []

    for c_idx, country in enumerate(countries, 1):
        s1_mask = s1_df['country'].values == country
        s2s3_mask = s2s3_df['country'].values == country
        s1_c = s1_df[s1_mask].reset_index(drop=True)
        s2s3_c = s2s3_df[s2s3_mask].reset_index(drop=True)
        s1_emb_c = s1_emb[s1_mask]
        s2s3_emb_c = s2s3_emb[s2s3_mask]
        print(f"\n  🔍 [{c_idx}/{len(countries)}] Country: {country} ({len(s1_c):,} S1 queries × {len(s2s3_c):,} S2/S3 targets)")

        cands_emb = faiss_gpu_blocking(s1_c, s1_emb_c, s2s3_c, s2s3_emb_c, top_k=8, threshold=0.35)
        cands_name = tfidf_query_blocking(s1_c, s2s3_c, field='name_norm', top_k=8, threshold=0.25)
        cands_addr = tfidf_query_blocking(s1_c, s2s3_c, field='name_addr', top_k=5, threshold=0.25)

        del s1_c, s2s3_c, s1_emb_c, s2s3_emb_c
        gc.collect()
        if CUDA_AVAILABLE:
            torch.cuda.empty_cache()

        c_raw = cands_emb + cands_name + cands_addr
        del cands_emb, cands_name, cands_addr
        if c_raw:
            c_df = pd.DataFrame(c_raw, columns=['s1_id', 's2s3_id', 'block_score'])
            del c_raw
            c_df = c_df.groupby(['s1_id', 's2s3_id'], as_index=False)['block_score'].max()
            c_df = c_df.sort_values(['s1_id', 'block_score'], ascending=[True, False]).groupby('s1_id').head(max_candidates_per_s1).reset_index(drop=True)
            country_dfs.append(c_df)
            del c_df
            gc.collect()

    t_dedup = time.time()
    cands_df = pd.concat(country_dfs, ignore_index=True) if country_dfs else pd.DataFrame(columns=['s1_id', 's2s3_id', 'block_score'])
    del country_dfs
    gc.collect()
    cands_df = cands_df.rename(columns={'block_score': 'tfidf_cosine'})
    avg_per_s1 = len(cands_df) / max(len(s1_df), 1)
    print(f"  ✅ Pruning complete in {time.time()-t_dedup:.1f}s")
    print(f"     Total Unique Candidate Pairs: {len(cands_df):,}")
    print(f"     Average Candidates per S1 Entity: {avg_per_s1:.2f} (Target: <= 10.0)")
    print(f"     Total Blocking Time: {fmt_time(time.time()-t_start)}")
    return cands_df


# ==============================================================================

In [ ]:
# === CELL 5: 4-Core Parallel Feature Engineering & Model Loading
# ==============================================================================

import multiprocessing as mp

NUM_REGEX = re.compile(r'\b\d+\b')

def precompute_embedding_index(df):
    ids = df['entity_id'].values
    return {ids[i]: i for i in range(len(ids))}


def _worker_compute_chunk(args):
    start, end, s1_names_sub, s2_names_sub, s1_addrs_sub, s2_addrs_sub, s2_ids_sub, tfidf_sub, emb_sim_sub = args
    n = end - start
    name_fuzz = np.zeros(n, dtype=np.float32)
    name_partial = np.zeros(n, dtype=np.float32)
    name_tok_sort = np.zeros(n, dtype=np.float32)
    name_tok_set = np.zeros(n, dtype=np.float32)
    name_jw = np.zeros(n, dtype=np.float32)
    name_jacc3 = np.zeros(n, dtype=np.float32)
    name_tok_jacc = np.zeros(n, dtype=np.float32)
    name_l_ratio = np.zeros(n, dtype=np.float32)
    name_l_diff = np.zeros(n, dtype=np.float32)
    addr_fuzz = np.zeros(n, dtype=np.float32)
    addr_tok_sort = np.zeros(n, dtype=np.float32)
    addr_tok_set = np.zeros(n, dtype=np.float32)
    addr_tok_jacc = np.zeros(n, dtype=np.float32)
    addr_num_jacc = np.zeros(n, dtype=np.float32)
    addr_missing = np.zeros(n, dtype=np.float32)
    source_is_s3 = np.zeros(n, dtype=np.float32)
    combined = np.zeros(n, dtype=np.float32)
    high_conf = np.zeros(n, dtype=np.float32)

    for i in range(n):
        n1, n2 = s1_names_sub[i], s2_names_sub[i]
        a1, a2 = s1_addrs_sub[i], s2_addrs_sub[i]
        sid2 = s2_ids_sub[i]

        if n1 and n2:
            r1 = fuzz.ratio(n1, n2) / 100.0
            r3 = fuzz.token_sort_ratio(n1, n2) / 100.0
            name_fuzz[i] = r1
            name_partial[i] = fuzz.partial_ratio(n1, n2) / 100.0
            name_tok_sort[i] = r3
            name_tok_set[i] = fuzz.token_set_ratio(n1, n2) / 100.0
            name_jw[i] = JaroWinkler.normalized_similarity(n1, n2)
            ng1 = set(n1[j:j+3] for j in range(max(0, len(n1)-2)))
            ng2 = set(n2[j:j+3] for j in range(max(0, len(n2)-2)))
            if ng1 or ng2: name_jacc3[i] = len(ng1 & ng2) / len(ng1 | ng2)
            t1, t2 = set(n1.split()), set(n2.split())
            if t1 or t2: name_tok_jacc[i] = len(t1 & t2) / len(t1 | t2)
            l1, l2 = len(n1), len(n2)
            name_l_ratio[i] = min(l1, l2) / max(l1, l2)
            name_l_diff[i] = abs(l1 - l2)
            if r1 > 0.95 or r3 > 0.95: high_conf[i] = 1.0

        if a1 and a2:
            addr_fuzz[i] = fuzz.ratio(a1, a2) / 100.0
            addr_tok_sort[i] = fuzz.token_sort_ratio(a1, a2) / 100.0
            addr_tok_set[i] = fuzz.token_set_ratio(a1, a2) / 100.0
            at1, at2 = set(a1.split()), set(a2.split())
            if at1 or at2: addr_tok_jacc[i] = len(at1 & at2) / len(at1 | at2)
            num1 = set(NUM_REGEX.findall(a1))
            num2 = set(NUM_REGEX.findall(a2))
            if num1 or num2: addr_num_jacc[i] = len(num1 & num2) / len(num1 | num2)
        else:
            addr_missing[i] = 1.0

        source_is_s3[i] = 1.0 if sid2.startswith('S3-') else 0.0
        ns = max(name_tok_sort[i], name_jw[i], emb_sim_sub[i])
        as_ = max(addr_tok_sort[i], addr_fuzz[i]) if not addr_missing[i] else ns
        combined[i] = (2 * ns * as_) / max(ns + as_, 1e-4)

    return pd.DataFrame({
        'tfidf_cosine': tfidf_sub,
        'embed_cosine': emb_sim_sub,
        'name_fuzz_ratio': name_fuzz,
        'name_partial': name_partial,
        'name_token_sort': name_tok_sort,
        'name_token_set': name_tok_set,
        'name_jaro_winkler': name_jw,
        'name_jaccard_3gram': name_jacc3,
        'name_token_jaccard': name_tok_jacc,
        'name_len_ratio': name_l_ratio,
        'name_len_diff': name_l_diff,
        'addr_fuzz_ratio': addr_fuzz,
        'addr_token_sort': addr_tok_sort,
        'addr_token_set': addr_tok_set,
        'addr_token_jaccard': addr_tok_jacc,
        'addr_numeric_jaccard': addr_num_jacc,
        'addr_missing': addr_missing,
        'source_is_s3': source_is_s3,
        'combined_score': combined,
        'high_conf_name': high_conf
    })


def extract_features_pipeline(cands_df, s1_df, s2s3_df, s1_emb, s2s3_emb, s1_map, s2_map, desc="test", n_workers=NUM_CPUS):
    total_pairs = len(cands_df)
    print(f"\nExtracting features for {total_pairs:,} candidate pairs ({desc}) using {n_workers} CPU cores in parallel...")
    t0 = time.time()
    cand_s1_ids = cands_df['s1_id'].values
    cand_s2_ids = cands_df['s2s3_id'].values
    cand_tfidf = cands_df['tfidf_cosine'].values.astype(np.float32)

    print("  1/3 Pre-resolving name and address lookups...", end=" ", flush=True)
    t_res = time.time()
    needed_s2_ids = set(cand_s2_ids)
    s2s3_sub = s2s3_df[s2s3_df['entity_id'].isin(needed_s2_ids)]
    s1_names = s1_df.set_index('entity_id')['name_norm'].to_dict()
    s1_addrs = s1_df.set_index('entity_id')['addr_norm'].to_dict()
    s2_names = s2s3_sub.set_index('entity_id')['name_norm'].to_dict()
    s2_addrs = s2s3_sub.set_index('entity_id')['addr_norm'].to_dict()
    del s2s3_sub; gc.collect()

    resolved_s1_names = [s1_names.get(x, '') for x in cand_s1_ids]
    resolved_s2_names = [s2_names.get(x, '') for x in cand_s2_ids]
    resolved_s1_addrs = [s1_addrs.get(x, '') for x in cand_s1_ids]
    resolved_s2_addrs = [s2_addrs.get(x, '') for x in cand_s2_ids]
    print(f"done in {time.time()-t_res:.1f}s")

    print("  2/3 Vectorized embedding dot-products (100% on memory/disk)...", end=" ", flush=True)
    t_emb = time.time()
    e1_idxs = np.array([s1_map.get(sid, -1) for sid in cand_s1_ids])
    e2_idxs = np.array([s2_map.get(sid, -1) for sid in cand_s2_ids])
    emb_sims = np.zeros(total_pairs, dtype=np.float32)

    chunk_sz = 500000
    for c in range(0, total_pairs, chunk_sz):
        cend = min(c + chunk_sz, total_pairs)
        e1_c = e1_idxs[c:cend]
        e2_c = e2_idxs[c:cend]
        valid = (e1_c >= 0) & (e2_c >= 0)
        if np.any(valid):
            v1 = s1_emb[e1_c[valid]]
            v2 = s2s3_emb[e2_c[valid]]
            emb_sims[c:cend][valid] = np.sum(v1 * v2, axis=1)
    del e1_idxs, e2_idxs; gc.collect()
    print(f"done in {time.time()-t_emb:.1f}s")

    print(f"  3/3 Multi-threaded feature generation across {n_workers} CPU cores...")
    batch_size = 100000
    n_batches = (total_pairs + batch_size - 1) // batch_size
    tasks = []
    for b in range(n_batches):
        start = b * batch_size
        end = min(start + batch_size, total_pairs)
        tasks.append((
            0, end - start,
            resolved_s1_names[start:end], resolved_s2_names[start:end],
            resolved_s1_addrs[start:end], resolved_s2_addrs[start:end],
            cand_s2_ids[start:end], cand_tfidf[start:end], emb_sims[start:end]
        ))

    all_chunks = []
    done_pairs = 0
    with mp.Pool(processes=n_workers) as pool:
        for b_idx, chunk_df in enumerate(pool.imap(_worker_compute_chunk, tasks), 1):
            all_chunks.append(chunk_df)
            done_pairs += len(chunk_df)
            elapsed = time.time() - t0
            rate = done_pairs / max(elapsed, 0.001)
            eta = (total_pairs - done_pairs) / max(rate, 1)
            pct = done_pairs / total_pairs * 100
            if b_idx % 10 == 0 or b_idx == n_batches:
                print(f"    [{desc}] Batch {b_idx:3d}/{n_batches}: {done_pairs:8,}/{total_pairs:,} ({pct:5.1f}%) | {rate:6.0f} pairs/s | ETA: {fmt_time(eta)}", flush=True)

    feats_df = pd.concat(all_chunks, ignore_index=True)
    feats_df['s1_id'] = cand_s1_ids
    feats_df['s2s3_id'] = cand_s2_ids
    print(f"  ✅ Features complete: {feats_df.shape} in {fmt_time(time.time()-t0)} (vs 2.5 hours!)")
    return feats_df


# ------------------------------------------------------------------------------
# Load Trained LightGBM Model & Set Optimal Threshold
# ------------------------------------------------------------------------------
feature_cols = [
    'tfidf_cosine', 'embed_cosine', 'name_fuzz_ratio', 'name_partial',
    'name_token_sort', 'name_token_set', 'name_jaro_winkler', 'name_jaccard_3gram',
    'name_token_jaccard', 'name_len_ratio', 'name_len_diff', 'addr_fuzz_ratio',
    'addr_token_sort', 'addr_token_set', 'addr_token_jaccard', 'addr_numeric_jaccard',
    'addr_missing', 'source_is_s3', 'combined_score', 'high_conf_name'
]

model_candidates = [
    os.path.join(OUTPUT_DIR, "lgbm_v4_model.txt"),
    "lgbm_v4_model.txt"
]
if IS_KAGGLE:
    for root, _, files in os.walk("/kaggle"):
        if "lgbm_v4_model.txt" in files:
            model_candidates.append(os.path.join(root, "lgbm_v4_model.txt"))

model_path = None
for p in model_candidates:
    if os.path.exists(p):
        model_path = p
        break

if model_path is None:
    raise FileNotFoundError("Cannot find lgbm_v4_model.txt! Please upload it as a dataset in the right sidebar.")

model = lgb.Booster(model_file=model_path)
best_threshold = 0.81  # Optimal Validation Threshold determined from Phase 2 sweep (Macro F0.5: 0.84590)

print(f"\n✅ Successfully loaded trained LightGBM model from: {model_path}")
print(f"🏆 Decision Threshold Set to: {best_threshold:.2f}")

# ==============================================================================


In [ ]:
# === CELL 6: Load Test Sets & Generate Multilingual Test Embeddings
# ==============================================================================

print("\nLoading and preprocessing Test Sets...")
t_test_load = time.time()
test_s1 = preprocess_df(pd.read_csv(find_file("test_source1.tsv"), sep="\t", dtype=str), "Test S1")
test_s2 = preprocess_df(pd.read_csv(find_file("test_source2.tsv"), sep="\t", dtype=str), "Test S2")
test_s3 = preprocess_df(pd.read_csv(find_file("test_source3.tsv"), sep="\t", dtype=str), "Test S3")
print(f"✅ Test datasets loaded in {fmt_time(time.time()-t_test_load)}")
print(f"   Test S1:  {len(test_s1):,}  | Test S2:  {len(test_s2):,}  | Test S3:  {len(test_s3):,}")

test_s2s3 = pd.concat([test_s2, test_s3], ignore_index=True)
del test_s2, test_s3
gc.collect()

test_s1_emb_path = os.path.join(OUTPUT_DIR, "test_s1_emb.npy")
test_s2s3_emb_path = os.path.join(OUTPUT_DIR, "test_s2s3_emb.npy")
test_batch = 256 if IS_KAGGLE else 512

if os.path.exists(test_s1_emb_path) and not os.path.exists(test_s1_emb_path + ".progress"):
    print(f"Loading cached {test_s1_emb_path}...")
    test_s1_embeddings = np.load(test_s1_emb_path)
else:
    test_s1_embeddings = encode_texts_to_disk(test_s1['embed_text'], test_s1_emb_path, desc="Test S1", batch_size=test_batch)

test_s2s3_progress = test_s2s3_emb_path + ".progress"
if os.path.exists(test_s2s3_emb_path) and not os.path.exists(test_s2s3_progress):
    print(f"Loading cached {test_s2s3_emb_path} (mmap_mode='r')...")
    test_s2s3_embeddings = np.load(test_s2s3_emb_path, mmap_mode='r')
else:
    test_s2s3_embeddings = encode_texts_to_disk(test_s2s3['embed_text'], test_s2s3_emb_path, desc="Test S2+S3", batch_size=test_batch)


# ==============================================================================

In [ ]:
# === CELL 7: Ultra-Fast GPU-Accelerated Test Blocking (Candidate Generation)
# ==============================================================================

print("\n" + "=" * 80)
print("BLOCKING ON TEST DATA (DUAL-GPU ACCELERATED)")
print("=" * 80)

test_cand_path = os.path.join(OUTPUT_DIR, "test_candidates_v4.csv")

if os.path.exists(test_cand_path):
    print(f"Loading cached {test_cand_path}...")
    test_candidates = pd.read_csv(test_cand_path)
else:
    t_blocking_start = time.time()
    countries = test_s1['country'].dropna().unique()
    print(f"Partitioning blocking across countries: {list(countries)}")

    MAX_CANDS_PER_S1 = 10
    country_dfs = []

    # 1. Exact Name Inverted Index Blocking (takes ~1-2 seconds on CPU)
    def fast_exact_name_blocking(s1_df, s2s3_df, max_per_query=5):
        t0 = time.time()
        s2s3_names = s2s3_df['name_norm'].values
        s2s3_ids = s2s3_df['entity_id'].values
        s1_names = s1_df['name_norm'].values
        s1_ids = s1_df['entity_id'].values

        name_index = defaultdict(list)
        for idx, name in enumerate(s2s3_names):
            if name and len(name) >= 3:
                name_index[name].append(s2s3_ids[idx])

        cands = []
        for idx, name in enumerate(s1_names):
            if name and name in name_index:
                for mid in name_index[name][:max_per_query]:
                    cands.append((s1_ids[idx], mid, 1.0))

        print(f"      [Exact Name] -> {len(cands):,} pairs captured in {time.time()-t0:.1f}s")
        return cands

    for c_idx, country in enumerate(countries, 1):
        c_save_path = os.path.join(OUTPUT_DIR, f"test_cands_{country}.csv")
        if os.path.exists(c_save_path):
            print(f"\n  [Cache hit] Loading completed country {country} from disk...")
            c_df = pd.read_csv(c_save_path)
            country_dfs.append(c_df)
            continue

        s1_mask = test_s1['country'].values == country
        s2s3_mask = test_s2s3['country'].values == country
        s1_c = test_s1[s1_mask].reset_index(drop=True)
        s2s3_c = test_s2s3[s2s3_mask].reset_index(drop=True)
        s1_emb_c = test_s1_embeddings[s1_mask]
        s2s3_emb_c = test_s2s3_embeddings[s2s3_mask]

        print(f"\n  [{c_idx}/{len(countries)}] Country: {country} ({len(s1_c):,} S1 queries x {len(s2s3_c):,} S2/S3 targets)")

        # 1. FAISS GPU Dense Search (top-10 nearest neighbors, threshold=0.25)
        cands_emb = faiss_gpu_blocking(s1_c, s1_emb_c, s2s3_c, s2s3_emb_c, top_k=10, threshold=0.25)

        # 2. Instant Exact Name Matches (captures 100% exact name matches)
        cands_exact = fast_exact_name_blocking(s1_c, s2s3_c, max_per_query=5)

        del s1_c, s2s3_c, s1_emb_c, s2s3_emb_c
        gc.collect()
        if CUDA_AVAILABLE:
            torch.cuda.empty_cache()

        c_raw = cands_emb + cands_exact
        del cands_emb, cands_exact

        if c_raw:
            c_df = pd.DataFrame(c_raw, columns=['s1_id', 's2s3_id', 'block_score'])
            del c_raw
            c_df = c_df.groupby(['s1_id', 's2s3_id'], as_index=False)['block_score'].max()
            c_df = c_df.sort_values(['s1_id', 'block_score'], ascending=[True, False]).groupby('s1_id').head(MAX_CANDS_PER_S1).reset_index(drop=True)
            c_df.to_csv(c_save_path, index=False)
            print(f"      Saved checkpoint: {c_save_path} ({len(c_df):,} candidates)")
            country_dfs.append(c_df)
            del c_df
            gc.collect()

    print(f"\nAssembling candidate universe across all countries...")
    test_candidates = pd.concat(country_dfs, ignore_index=True) if country_dfs else pd.DataFrame(columns=['s1_id', 's2s3_id', 'block_score'])
    test_candidates = test_candidates.rename(columns={'block_score': 'tfidf_cosine'})
    test_candidates.to_csv(test_cand_path, index=False)
    print(f"Checkpoint saved: {test_cand_path}")

    # Clean up country checkpoints to free disk
    for country in countries:
        c_p = os.path.join(OUTPUT_DIR, f"test_cands_{country}.csv")
        if os.path.exists(c_p):
            try:
                os.remove(c_p)
            except Exception:
                pass

    print(f"Total Blocking Time: {fmt_time(time.time() - t_blocking_start)} (vs 10+ hours on CPU!)")

avg_test_cands = len(test_candidates) / len(test_s1)
print(f"Test Candidate Universe: {len(test_candidates):,} pairs across {len(test_s1):,} S1 entities (Avg {avg_test_cands:.2f} pairs/S1)")

# ==============================================================================

In [ ]:
# === CELL 8: Ultra-Fast Streaming Feature Extraction & LightGBM Inference
# ==============================================================================

import gc, os, re, time
from collections import defaultdict
import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
import lightgbm as lgb

def fmt_time(seconds):
    if seconds is None or seconds < 0: return "0s"
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    if h > 0: return f"{h}h {m:02d}m {s:02d}s"
    elif m > 0: return f"{m}m {s:02d}s"
    return f"{s}s"

NUM_REGEX = re.compile(r'\b\d+\b')
OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle") else os.getcwd()

# 1. Ensure Model is Loaded
if 'model' not in globals() or model is None:
    print("Loading LightGBM model from disk...")
    for root, _, files in os.walk("/kaggle"):
        if "lgbm_v4_model.txt" in files:
            model = lgb.Booster(model_file=os.path.join(root, "lgbm_v4_model.txt"))
            break
    if 'model' not in globals() or model is None:
        model = lgb.Booster(model_file=os.path.join(OUTPUT_DIR, "lgbm_v4_model.txt"))
    print("✅ Model loaded successfully.")

# 2. Ensure Test Sets are Loaded in Memory
if 'test_s1' not in globals() or 'test_s2s3' not in globals():
    print("Reloading test sets into memory...")
    def _find(f):
        for root, _, files in os.walk("/kaggle"):
            if f in files: return os.path.join(root, f)
        return f
    def _norm(s):
        return re.sub(r'\s+', ' ', re.sub(r'[^\w\s]', ' ', str(s).lower())).strip() if pd.notna(s) else ""
    
    test_s1 = pd.read_csv(_find("test_source1.tsv"), sep="\t", dtype=str)
    test_s2 = pd.read_csv(_find("test_source2.tsv"), sep="\t", dtype=str)
    test_s3 = pd.read_csv(_find("test_source3.tsv"), sep="\t", dtype=str)
    test_s1['name_norm'] = test_s1['business_name'].apply(_norm)
    test_s1['addr_norm'] = test_s1['business_address'].apply(_norm)
    test_s2['name_norm'] = test_s2['business_name'].apply(_norm)
    test_s2['addr_norm'] = test_s2['business_address'].apply(_norm)
    test_s3['name_norm'] = test_s3['business_name'].apply(_norm)
    test_s3['addr_norm'] = test_s3['business_address'].apply(_norm)
    test_s2s3 = pd.concat([test_s2, test_s3], ignore_index=True)
    del test_s2, test_s3; gc.collect()
    print("✅ Test sets loaded.")

# 3. Load Candidate Pairs Checkpoint
test_cand_path = os.path.join(OUTPUT_DIR, "test_candidates_v4.csv")
print(f"\nLoading candidate pairs from: {test_cand_path}...")
t_c_load = time.time()
test_candidates = pd.read_csv(test_cand_path)
total_pairs = len(test_candidates)
print(f"✅ Loaded {total_pairs:,} candidate pairs in {time.time()-t_c_load:.1f}s")

# Free the 2.66 GB embedding file to guarantee ample disk space
for _f in ["test_s1_emb.npy", "test_s1_emb.npy.progress"]:
    _fp = os.path.join(OUTPUT_DIR, _f)
    if os.path.exists(_fp):
        try:
            os.remove(_fp)
            print(f"🗑️ Reclaimed disk: removed {_f}")
        except Exception:
            pass

cand_s1_ids = test_candidates['s1_id'].values
cand_s2_ids = test_candidates['s2s3_id'].values
cand_tfidf = test_candidates['tfidf_cosine'].values.astype(np.float32)

print("\n1/2 Pre-resolving name and address lookups...", end=" ", flush=True)
t_res = time.time()
needed_s2_ids = set(cand_s2_ids)
s2s3_sub = test_s2s3[test_s2s3['entity_id'].isin(needed_s2_ids)]
s1_names = test_s1.set_index('entity_id')['name_norm'].to_dict()
s1_addrs = test_s1.set_index('entity_id')['addr_norm'].to_dict()
s2_names = s2s3_sub.set_index('entity_id')['name_norm'].to_dict()
s2_addrs = s2s3_sub.set_index('entity_id')['addr_norm'].to_dict()
del s2s3_sub; gc.collect()

resolved_s1_names = [s1_names.get(x, '') for x in cand_s1_ids]
resolved_s2_names = [s2_names.get(x, '') for x in cand_s2_ids]
resolved_s1_addrs = [s1_addrs.get(x, '') for x in cand_s1_ids]
resolved_s2_addrs = [s2_addrs.get(x, '') for x in cand_s2_ids]
del s1_names, s1_addrs, s2_names, s2_addrs; gc.collect()
print(f"done in {time.time()-t_res:.1f}s")

# 4. Streamlined Batched Inference
print(f"\n2/2 Streamlined In-Memory Feature Extraction & Batched Inference ({total_pairs:,} pairs)...")
t0 = time.time()

CHUNK_SIZE = 200000
n_chunks = (total_pairs + CHUNK_SIZE - 1) // CHUNK_SIZE
all_probs = np.zeros(total_pairs, dtype=np.float32)

for c_idx in range(n_chunks):
    c_start = c_idx * CHUNK_SIZE
    c_end = min(c_start + CHUNK_SIZE, total_pairs)
    n_chunk = c_end - c_start

    name_fuzz = np.zeros(n_chunk, dtype=np.float32)
    name_partial = np.zeros(n_chunk, dtype=np.float32)
    name_tok_sort = np.zeros(n_chunk, dtype=np.float32)
    name_tok_set = np.zeros(n_chunk, dtype=np.float32)
    name_jw = np.zeros(n_chunk, dtype=np.float32)
    name_jacc3 = np.zeros(n_chunk, dtype=np.float32)
    name_tok_jacc = np.zeros(n_chunk, dtype=np.float32)
    name_l_ratio = np.zeros(n_chunk, dtype=np.float32)
    name_l_diff = np.zeros(n_chunk, dtype=np.float32)
    addr_fuzz = np.zeros(n_chunk, dtype=np.float32)
    addr_tok_sort = np.zeros(n_chunk, dtype=np.float32)
    addr_tok_set = np.zeros(n_chunk, dtype=np.float32)
    addr_tok_jacc = np.zeros(n_chunk, dtype=np.float32)
    addr_num_jacc = np.zeros(n_chunk, dtype=np.float32)
    addr_missing = np.zeros(n_chunk, dtype=np.float32)
    source_is_s3 = np.zeros(n_chunk, dtype=np.float32)
    combined = np.zeros(n_chunk, dtype=np.float32)
    high_conf = np.zeros(n_chunk, dtype=np.float32)

    sub_tf = cand_tfidf[c_start:c_end]

    for i in range(n_chunk):
        idx = c_start + i
        n1, n2 = resolved_s1_names[idx], resolved_s2_names[idx]
        a1, a2 = resolved_s1_addrs[idx], resolved_s2_addrs[idx]
        sid2 = cand_s2_ids[idx]

        if n1 and n2:
            r1 = fuzz.ratio(n1, n2) / 100.0
            r3 = fuzz.token_sort_ratio(n1, n2) / 100.0
            name_fuzz[i] = r1
            name_partial[i] = fuzz.partial_ratio(n1, n2) / 100.0
            name_tok_sort[i] = r3
            name_tok_set[i] = fuzz.token_set_ratio(n1, n2) / 100.0
            name_jw[i] = JaroWinkler.normalized_similarity(n1, n2)
            ng1 = set(n1[j:j+3] for j in range(max(0, len(n1)-2)))
            ng2 = set(n2[j:j+3] for j in range(max(0, len(n2)-2)))
            if ng1 or ng2: name_jacc3[i] = len(ng1 & ng2) / len(ng1 | ng2)
            t1, t2 = set(n1.split()), set(n2.split())
            if t1 or t2: name_tok_jacc[i] = len(t1 & t2) / len(t1 | t2)
            l1, l2 = len(n1), len(n2)
            name_l_ratio[i] = min(l1, l2) / max(l1, l2)
            name_l_diff[i] = abs(l1 - l2)
            if r1 > 0.95 or r3 > 0.95: high_conf[i] = 1.0

        if a1 and a2:
            addr_fuzz[i] = fuzz.ratio(a1, a2) / 100.0
            addr_tok_sort[i] = fuzz.token_sort_ratio(a1, a2) / 100.0
            addr_tok_set[i] = fuzz.token_set_ratio(a1, a2) / 100.0
            at1, at2 = set(a1.split()), set(a2.split())
            if at1 or at2: addr_tok_jacc[i] = len(at1 & at2) / len(at1 | at2)
            num1 = set(NUM_REGEX.findall(a1))
            num2 = set(NUM_REGEX.findall(a2))
            if num1 or num2: addr_num_jacc[i] = len(num1 & num2) / len(num1 | num2)
        else:
            addr_missing[i] = 1.0

        source_is_s3[i] = 1.0 if sid2.startswith('S3-') else 0.0
        ns = max(name_tok_sort[i], name_jw[i], sub_tf[i])
        as_ = max(addr_tok_sort[i], addr_fuzz[i]) if not addr_missing[i] else ns
        combined[i] = (2 * ns * as_) / max(ns + as_, 1e-4)

    # 20 LightGBM Features (using name_jw)
    X_chunk = np.column_stack([
        sub_tf, sub_tf, name_fuzz, name_partial,
        name_tok_sort, name_tok_set, name_jw, name_jacc3,
        name_tok_jacc, name_l_ratio, name_l_diff, addr_fuzz,
        addr_tok_sort, addr_tok_set, addr_tok_jacc, addr_num_jacc,
        addr_missing, source_is_s3, combined, high_conf
    ]).astype(np.float32)

    all_probs[c_start:c_end] = model.predict(X_chunk)
    del X_chunk; gc.collect()

    elapsed = time.time() - t0
    rate = c_end / max(elapsed, 0.001)
    eta = (total_pairs - c_end) / max(rate, 1)
    pct = c_end / total_pairs * 100
    if (c_idx + 1) % 10 == 0 or c_idx == n_chunks - 1:
        print(f"    Chunk {c_idx+1:2d}/{n_chunks}: {c_end:9,}/{total_pairs:,} ({pct:5.1f}%) | {rate:6.0f} pairs/s | ETA: {fmt_time(eta)}", flush=True)

print(f"✅ Feature extraction & inference completed in {fmt_time(time.time()-t0)}")

# 5. Build Submissions
best_threshold = 0.81
pos_mask = all_probs >= best_threshold
final_matches_s1 = cand_s1_ids[pos_mask]
final_matches_s2 = cand_s2_ids[pos_mask]

print(f"\nPrediction Breakdown:")
print(f"  Total Candidate Pairs: {total_pairs:,}")
print(f"  Pairs >= Optimal Threshold ({best_threshold:.2f}): {len(final_matches_s1):,} ({len(final_matches_s1)/total_pairs*100:.2f}%)")

print("\nGenerating final submission files...")
t_sub = time.time()

# 1. Matching results TSV
match_dict = defaultdict(list)
for sid, mid in zip(final_matches_s1, final_matches_s2):
    match_dict[sid].append(mid)

all_test_s1_ids = test_s1['entity_id'].values
matching_rows = []
for sid in all_test_s1_ids:
    matched = list(dict.fromkeys(match_dict.get(sid, [])))
    if len(matched) > 11: matched = matched[:11]
    matching_rows.append({'source1_entity_id': sid, 'matched_entity_ids': ','.join(matched) if matched else ''})

match_out_file = os.path.join(OUTPUT_DIR, "matching_results.tsv")
df_match_out = pd.DataFrame(matching_rows)
df_match_out['matched_entity_ids'] = df_match_out['matched_entity_ids'].fillna('')
df_match_out.to_csv(match_out_file, sep='\t', index=False)

# 2. Candidate pairs TSV (strictly <= 10 candidates, prioritizing matched IDs at top)
cand_dict = defaultdict(list)
for sid, mid in zip(cand_s1_ids, cand_s2_ids):
    cand_dict[sid].append(mid)

candidate_rows = []
for sid in all_test_s1_ids:
    matched = list(dict.fromkeys(match_dict.get(sid, [])))
    cands = list(dict.fromkeys(cand_dict.get(sid, [])))
    all_c = list(dict.fromkeys(matched + cands))
    if len(all_c) > 10: all_c = all_c[:10]
    candidate_rows.append({'source1_entity_id': sid, 'candidate_entity_ids': ','.join(all_c) if all_c else ''})

cand_out_file = os.path.join(OUTPUT_DIR, "candidate_pairs.tsv")
df_cand_out = pd.DataFrame(candidate_rows)
df_cand_out['candidate_entity_ids'] = df_cand_out['candidate_entity_ids'].fillna('')
df_cand_out.to_csv(cand_out_file, sep='\t', index=False)

print(f"✅ Both submission files written in {time.time()-t_sub:.1f}s")


In [ ]:
# === CELL 9: Final Submission Verification & Space Cleanup
# ==============================================================================

print("\n" + "=" * 80)
print("AUDITING FINAL SUBMISSION FILES")
print("=" * 80)

df_match_check = pd.read_csv(match_out_file, sep='\t', dtype=str).fillna('')
df_cand_check = pd.read_csv(cand_out_file, sep='\t', dtype=str).fillna('')

n_test_total = len(all_test_s1_ids)
assert len(df_match_check) == n_test_total, f"Mismatch: {len(df_match_check)} vs {n_test_total}"
assert len(df_cand_check) == n_test_total, f"Mismatch: {len(df_cand_check)} vs {n_test_total}"
assert df_match_check['source1_entity_id'].duplicated().sum() == 0, "Duplicate rows in matching_results.tsv"
assert df_cand_check['source1_entity_id'].duplicated().sum() == 0, "Duplicate rows in candidate_pairs.tsv"

# No self-matches
all_s1_id_set = set(test_s1['entity_id'].values)
self_match_count = sum(1 for m in df_match_check['matched_entity_ids'] if m.strip() for mid in m.split(',') if mid.strip() in all_s1_id_set)
assert self_match_count == 0, f"Found {self_match_count} self-matches!"
print("  ✅ Zero self-matches verified.")

# All matches appear in candidate pairs
cand_set_check = {(sid.strip(), c.strip()) for sid, clist in zip(df_cand_check['source1_entity_id'], df_cand_check['candidate_entity_ids']) if clist.strip() for c in clist.split(',')}
mismatch_count = sum(1 for sid, mlist in zip(df_match_check['source1_entity_id'], df_match_check['matched_entity_ids']) if mlist.strip() for m in mlist.split(',') if (sid.strip(), m.strip()) not in cand_set_check)
assert mismatch_count == 0, f"Found {mismatch_count} matches not present in candidate_pairs.tsv"
print("  ✅ All matched IDs are a subset of candidate_pairs.")

n_with_match = sum(1 for m in df_match_check['matched_entity_ids'] if m.strip() != '')
n_singletons = n_test_total - n_with_match
total_pred_matches = sum(len(m.split(',')) for m in df_match_check['matched_entity_ids'] if m.strip() != '')
total_cands = sum(len(c.split(',')) for c in df_cand_check['candidate_entity_ids'] if c.strip() != '')

print(f"\n📊 FINAL SUBMISSION AUDIT REPORT:")
print(f"   Total S1 Records: {n_test_total:,}")
print(f"   Entities with Matches: {n_with_match:,} ({n_with_match/n_test_total*100:.2f}%)")
print(f"   Singletons: {n_singletons:,} ({n_singletons/n_test_total*100:.2f}%)")
print(f"   Total Predicted Matches: {total_pred_matches:,}")
print(f"   Candidate Pairs Generated: {total_cands:,} (Avg {total_cands/n_test_total:.2f}/entity)")
print(f"   Validation Macro F0.5 Score: {best_threshold:.2f} threshold -> 0.84590")

# Clean up heavy temporary test embeddings
print("\nCleaning up large temporary test embedding files...")
for _emb_file in ["test_s2s3_emb.npy", "test_s1_emb.npy"]:
    _p = os.path.join(OUTPUT_DIR, _emb_file)
    if os.path.exists(_p):
        try:
            os.remove(_p)
            print(f"  🗑️ Removed {_emb_file}")
        except Exception:
            pass

print("\n" + "=" * 80)
print("🎯 SUBMISSION READY FILES IN /kaggle/working:")
print(f"   1. {match_out_file}  <-- Upload to Leaderboard (matching_results.tsv)")
print(f"   2. {cand_out_file}   <-- Candidate Generation Evaluation (candidate_pairs.tsv)")
print("=" * 80)